# 📓 Notebook 2: Chunking & Pre-processing

**Purpose**: Break structured text into LLM-sized chunks, generate a glossary of key terms, and create a book-level summary for cross-chunk consistency.

**Pipeline**:
```
book_structure.json → Glossary extraction → Book summary → Paragraph-level chunking → Stats → Save
```

**Inputs**: `data/intermediate/book_structure.json` (from Notebook 1)  
**Outputs**: `chunks.json`, `glossary.json`, `book_summary.txt`

In [ ]:
# ── Imports & Configuration ─────────────────────────────────────────────────
import sys
import json
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    BOOK_STRUCTURE_FILE, CHUNKS_FILE, GLOSSARY_FILE, BOOK_SUMMARY_FILE,
    MAX_CHUNK_TOKENS, OVERLAP_TOKENS, MODEL_NAME, OLLAMA_BASE_URL,
)
from src.chunker import (
    create_chunks, extract_glossary, generate_book_summary, get_chunk_stats,
)
from src.llm_client import OllamaClient

In [ ]:
# ── Load Book Structure ──────────────────────────────────────────────────────
with open(BOOK_STRUCTURE_FILE, "r", encoding="utf-8") as f:
    chapters = json.load(f)

print(f"📚 Loaded {len(chapters)} chapters from {BOOK_STRUCTURE_FILE.name}")

In [ ]:
# ── Initialize LLM Client ───────────────────────────────────────────────────
# Needed for glossary extraction and book summary generation.

llm = OllamaClient(model_name=MODEL_NAME, base_url=OLLAMA_BASE_URL)
assert llm.is_available(), f"❌ Ollama not reachable or model '{MODEL_NAME}' not pulled. Run: ollama pull {MODEL_NAME}"
print(f"✅ Connected to Ollama — model: {MODEL_NAME}")

In [ ]:
# ── Extract Glossary ─────────────────────────────────────────────────────────
# Identifies key terms and jargon, produces plain-English definitions.
# This glossary is injected into every simplification prompt for consistency.

print("📖 Extracting glossary of key terms...")
glossary = extract_glossary(chapters, llm_client=llm)

print(f"\n📝 Found {len(glossary)} key terms:")
for term, defn in list(glossary.items())[:10]:  # preview first 10
    print(f"   • {term}: {defn}")
if len(glossary) > 10:
    print(f"   ... and {len(glossary) - 10} more")

In [ ]:
# ── Generate Book Summary ────────────────────────────────────────────────────
# Creates a ~500-token overview of the book's topic and themes.
# Prepended to every simplification prompt for global context.

print("📝 Generating book summary...")
book_summary = generate_book_summary(chapters, llm_client=llm)

print(f"\n📋 Book Summary ({len(book_summary)} chars):")
print(book_summary)

In [ ]:
# ── Create Chunks ────────────────────────────────────────────────────────────
# Split chapters into LLM-sized chunks at paragraph boundaries.
# Respects chapter boundaries (never merges text from different chapters).

print(f"✂️  Chunking with max_tokens={MAX_CHUNK_TOKENS}, overlap={OVERLAP_TOKENS}...")
chunks = create_chunks(chapters, max_tokens=MAX_CHUNK_TOKENS, overlap_tokens=OVERLAP_TOKENS)

# ── Stats ─────────────────
stats = get_chunk_stats(chunks)
print(f"\n📊 Chunk Statistics:")
print(f"   Total chunks:     {stats['total_chunks']}")
print(f"   Total tokens:     {stats['total_tokens']:,}")
print(f"   Avg tokens/chunk: {stats['avg_tokens']:.0f}")
print(f"   Range:            {stats['min_tokens']} – {stats['max_tokens']}")
print(f"   Chapters covered: {stats['chapters_covered']}")
print(f"   ⏱️  Estimated processing time: ~{stats['estimated_minutes']:.0f} minutes")

In [ ]:
# ── Save Everything ──────────────────────────────────────────────────────────

with open(CHUNKS_FILE, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)
print(f"💾 Saved {len(chunks)} chunks → {CHUNKS_FILE.name}")

with open(GLOSSARY_FILE, "w", encoding="utf-8") as f:
    json.dump(glossary, f, ensure_ascii=False, indent=2)
print(f"💾 Saved glossary → {GLOSSARY_FILE.name}")

with open(BOOK_SUMMARY_FILE, "w", encoding="utf-8") as f:
    f.write(book_summary)
print(f"💾 Saved summary → {BOOK_SUMMARY_FILE.name}")

print(f"\n✅ Chunking complete. Proceed to 03_simplify.ipynb")